<div style="padding:22px;border-radius:16px;background:linear-gradient(120deg,#312e81,#0891b2);color:white">
<h1 style="margin:0">TD Learning with Q(s, a): Greedy Bootstrap</h1>
<p style="margin:8px 0 0;font-size:17px">A compact, exercise-first notebook using the max next-action value in a 4×4 gridworld.</p>
</div>

**Scope:** one-step Q updates with the target $r+\gamma\max_{a'}Q(s',a')$.

By the end, you will have assembled the complete greedy TD update used by Q-learning.

<div style="border-left:6px solid #f59e0b;background:#fffbeb;padding:12px 16px;border-radius:8px">
<b>Gridworld</b><br>States are numbered 0–15. States <b>0</b> and <b>15</b> are terminal. Every move gives reward <b>−1</b>. A move into a wall leaves the agent in the same state.
</div>

```text
 0T   1    2    3
 4    5    6    7
 8    9   10   11
12   13   14   15T
```

The behavior policy chooses `up`, `down`, `right`, or `left` uniformly at random to generate experience. The learning target still uses the largest next-state action value.

In [ ]:
import random

ROWS, COLS = 4, 4
STATES = list(range(ROWS * COLS))
TERMINALS = {0, 15}
ACTIONS = ("up", "down", "right", "left")
DELTAS = {"up": (-1, 0), "down": (1, 0),
          "right": (0, 1), "left": (0, -1)}

def step(state, action):
    """Apply one deterministic gridworld transition."""
    if state in TERMINALS:
        return state, 0, True
    row, col = divmod(state, COLS)
    dr, dc = DELTAS[action]
    new_row = min(max(row + dr, 0), ROWS - 1)
    new_col = min(max(col + dc, 0), COLS - 1)
    next_state = new_row * COLS + new_col
    return next_state, -1, next_state in TERMINALS

def behavior_action(state, rng):
    """Sample an exploratory action uniformly at random."""
    return rng.choice(ACTIONS)

def reset(rng):
    """Start in a uniformly sampled nonterminal state."""
    return rng.choice([s for s in STATES if s not in TERMINALS])

def new_q_table():
    """Return a zero-filled Q table: Q[state][action]."""
    return {s: {a: 0.0 for a in ACTIONS} for s in STATES}

def show_q_values(Q):
    """Print the four action values learned for every nonterminal state."""
    print("state |     up    down   right    left")
    print("-" * 42)
    for state in STATES:
        if state in TERMINALS:
            print(f"{state:>5} | terminal")
        else:
            values = " ".join(f"{Q[state][a]:7.2f}" for a in ACTIONS)
            print(f"{state:>5} | {values}")

# Quick environment checks
assert step(5, "right") == (6, -1, False)
assert step(1, "left") == (0, -1, True)
assert step(1, "up") == (1, -1, False)
print("✓ Gridworld ready")

<div style="border-left:6px solid #14b8a6;background:#f0fdfa;padding:12px 16px;border-radius:8px">
<b>Key idea — bootstrap after one step</b><br>Update the experienced state–action pair using the reward plus the largest current Q estimate available at the next state.
</div>

For a transition $(S_t,A_t) \rightarrow S_{t+1}$ with reward $R_{t+1}$:

$$\text{TD target}=R_{t+1}+\gamma\max_{a'}Q(S_{t+1},a')$$

$$\delta_t=R_{t+1}+\gamma\max_{a'}Q(S_{t+1},a')-Q(S_t,A_t)$$

$$Q(S_t,A_t)\leftarrow Q(S_t,A_t)+\alpha\delta_t$$

If $S_{t+1}$ is terminal, its future value is zero. Using the current maximum Q estimate to update another Q estimate is **bootstrapping**.

### Worked numerical example

Suppose action `right` moves the agent from state 5 to state 6 with reward $-1$. Let

$$Q(5,\text{right})=-2,$$

and at state 6 let the four action values be $-3,-2,-4,-3$. The greedy bootstrap is

$$\max_{a'}Q(6,a')=-2.$$

With $\gamma=1$ and $\alpha=0.1$:

$$\text{target}=-1+1(-2)=-3, \qquad \delta=-3-(-2)=-1,$$

$$Q(5,\text{right})\leftarrow -2+0.1(-1)=\boxed{-2.1}.$$

It is a **temporal difference** because it compares the current prediction $Q(S_t,A_t)$ with a new prediction built from information one time step later.

<table style="width:100%;border-collapse:collapse">
<tr style="background:#ede9fe"><th style="padding:9px">Monte Carlo</th><th style="padding:9px">TD(0)</th></tr>
<tr><td style="padding:9px">Waits until the episode ends</td><td style="padding:9px">Updates after each transition</td></tr>
<tr><td style="padding:9px">Target is the complete sampled return $G_t$</td><td style="padding:9px">Target is $R_{t+1}+\gamma\max_{a'}Q(S_{t+1},a')$</td></tr>
<tr><td style="padding:9px">Does not bootstrap</td><td style="padding:9px">Bootstraps from the next-state estimate</td></tr>
</table>

**Memory hook:** MC says “wait and see”; TD says “learn now from one step plus my current Q prediction.”

<div style="margin-top:12px;padding:10px 14px;background:#fff7ed;border-radius:8px;border:1px solid #fdba74"><b>Important:</b> using the max target makes this the core Q-learning update. The action used to gather experience can still be exploratory.</div>

<div style="padding:13px 16px;border-radius:10px;background:#eff6ff;border:1px solid #93c5fd">
<b>How to work through the exercises</b><br>Replace only the lines marked <code>TODO</code>, then run the checker immediately below. A checker prints a green pass or a focused hint; it does not reveal the answer.
</div>

In [ ]:
def check_close(name, actual, expected, tolerance=1e-10):
    """Small friendly checker used throughout the notebook."""
    passed = (actual is not None and abs(actual - expected) <= tolerance)
    icon = "✅" if passed else "🟠"
    detail = "passed" if passed else f"expected {expected!r}, got {actual!r}"
    print(f"{icon} {name}: {detail}")
    return passed

## Exercise 1 — Find the greedy next Q estimate

Select the largest of the four $Q(s',a')$ values. A terminal state contributes zero.

In [ ]:
def max_next_q(Q, next_state, done):
    """Compute the largest available Q value at the next state.

    Inputs:
        Q (dict): Nested table accessed as Q[state][action].
        next_state (int): State reached by the transition.
        done (bool): True when next_state is terminal.

    Output:
        float: Zero if terminal; otherwise max Q(next_state, action).

    Examples:
        If Q[6] contains -3, -2, -4, -3, return -2.0.
        max_next_q(Q, 15, True) -> 0.0

    Implement:
        Select the maximum action value at next_state.

    Why:
        The greedy bootstrap estimates the best future value currently known.
    """
    if done:
        return 0.0
    action_values = [Q[next_state][action] for action in ACTIONS]
    best_value = None  # TODO: take the maximum of action_values
    return best_value

In [ ]:
print("Exercise 1 checks")
Q_test = new_q_table()
Q_test[6].update({"up": -3.0, "down": -2.0, "right": -4.0, "left": -3.0})
check_close("greedy next value", max_next_q(Q_test, 6, False), -2.0)
check_close("terminal future value", max_next_q(Q_test, 15, True), 0.0)

## Exercise 2 — Build the target and TD error

Combine the reward with Exercise 1's maximum bootstrap, then compare the target with the current $Q(s,a)$.

In [ ]:
def td_error(Q, state, action, reward, next_state, gamma, done):
    """Compute the one-step temporal-difference error for Q.

    Inputs:
        Q (dict): Nested action-value table Q[state][action].
        state (int): State before the transition.
        action (str): Action taken in state.
        reward (float): Reward observed after that action.
        next_state (int): State reached by the transition.
        gamma (float): Discount factor.
        done (bool): Whether next_state is terminal.

    Output:
        float: TD target minus Q[state][action].

    Example:
        For the worked example, return -1.0 because target=-3 and Q(5,right)=-2.

    Implement:
        Compute reward + gamma * max_next_q(...), then subtract Q(s,a).

    Why:
        The error's sign gives the update direction and its size the correction.
    """
    next_estimate = max_next_q(Q, next_state, done)
    target = None  # TODO: reward plus discounted next_estimate
    if target is None:
        return None
    error = None  # TODO: target minus the current Q(state, action)
    return error

In [ ]:
print("Exercise 2 checks")
Q_error_test = new_q_table()
Q_error_test[5]["right"] = -2.0
Q_error_test[6].update({"up": -3.0, "down": -2.0, "right": -4.0, "left": -3.0})
check_close("worked-example error",
            td_error(Q_error_test, 5, "right", -1, 6, 1.0, False), -1.0)
check_close("terminal error",
            td_error(Q_error_test, 14, "right", -1, 15, 1.0, True), -1.0)

## Exercise 3 — Perform one TD(0) update

Now apply the TD error to exactly one entry: $Q(s,a)$.

In [ ]:
def td_update(Q, state, action, reward, next_state, alpha, gamma, done):
    """Apply one in-place TD(0) action-value update.

    Inputs:
        Q (dict): Action-value table, modified in place.
        state (int): State before the transition.
        action (str): Action taken; Q[state][action] is updated.
        reward (float): Reward from this transition.
        next_state (int): State reached after the action.
        alpha (float): Learning rate.
        gamma (float): Discount factor.
        done (bool): Whether next_state is terminal.

    Output:
        float: The TD error used for the update.

    Example:
        In the worked example, the function returns -1 and changes
        Q[5]["right"] from -2.0 to -2.1 when alpha is 0.1.

    Implement:
        Call td_error, then update only Q[state][action].

    Why:
        This is the complete learning operation for one transition.
    """
    error = None  # TODO: call td_error(...) with this transition
    if error is None:
        return None
    # TODO: move Q[state][action] by alpha times error
    return error

In [ ]:
print("Exercise 3 checks")
Q_update_test = new_q_table()
Q_update_test[5]["right"] = -2.0
Q_update_test[6].update({"up": -3.0, "down": -2.0, "right": -4.0, "left": -3.0})
observed_error = td_update(Q_update_test, 5, "right", -1, 6, 0.1, 1.0, False)
check_close("worked-example error", observed_error, -1.0)
check_close("worked-example Q(5,right)", Q_update_test[5]["right"], -2.1)

Q_terminal_test = new_q_table()
Q_terminal_test[14]["right"] = -0.4
td_update(Q_terminal_test, 14, "right", -1, 15, 0.5, 1.0, True)
check_close("terminal transition", Q_terminal_test[14]["right"], -0.7)

## Exercise 4 — Learn online through one episode

Sample an exploratory action, update its Q entry immediately with the greedy next-state target, move to the next state, and repeat.

In [ ]:
def td0_episode(Q, alpha, gamma, rng, start_state=None, max_steps=500):
    """Run one exploratory episode and apply greedy TD updates online.

    Inputs:
        Q (dict): Action values, modified after every transition.
        alpha (float): Learning rate.
        gamma (float): Discount factor.
        rng (random.Random-like): Supplies choice(sequence) for reproducibility.
        start_state (int | None): Optional nonterminal start; random if None.
        max_steps (int): Safety limit for an unusually long random episode.

    Output:
        int: Number of transitions taken.

    Example:
        Starting in state 1 and choosing left ends in one transition, so return 1.

    Implement:
        Call td_update with the observed state, action, reward, and next state.

    Why:
        This turns the one-step rule into online learning across an episode.
    """
    state = reset(rng) if start_state is None else start_state
    steps = 0

    while state not in TERMINALS and steps < max_steps:
        action = behavior_action(state, rng)
        next_state, reward, done = step(state, action)
        # TODO: call td_update(...) using Q and this transition
        state = next_state
        steps += 1
        if done:
            break

    return steps

In [ ]:
class AlwaysLeft:
    """Tiny deterministic policy source for the checker."""
    def choice(self, sequence):
        return "left"

print("Exercise 4 checks")
Q_episode_test = new_q_table()
episode_steps = td0_episode(
    Q_episode_test, alpha=0.2, gamma=1.0,
    rng=AlwaysLeft(), start_state=1
)
check_close("one-step episode length", episode_steps, 1)
check_close("online Q(1,left) update", Q_episode_test[1]["left"], -0.2)
check_close("untaken Q(1,right) unchanged", Q_episode_test[1]["right"], 0.0)

## Exercise 5 — Assemble repeated greedy Q updates

Repeat exploratory episodes so experience gradually shapes the action-value table. Terminal action values remain zero.

In [ ]:
def q_learning(num_episodes, alpha=0.1, gamma=1.0, seed=7):
    """Learn Q values with exploratory behavior and a greedy TD target.

    Inputs:
        num_episodes (int): Number of sampled episodes.
        alpha (float): Constant learning rate.
        gamma (float): Discount factor; use 1.0 for this exercise.
        seed (int): Random seed for reproducible starts and actions.

    Output:
        dict: Q[state][action] for all sixteen states and four actions.

    Example:
        q_learning(0)[5]["right"] -> 0.0
        After training, visited nonterminal action values should be negative.

    Implement:
        Run td0_episode once per episode using the shared Q table and rng.

    Why:
        Reusing Q makes later experience refine earlier predictions.
    """
    Q = new_q_table()
    rng = random.Random(seed)

    for _ in range(num_episodes):
        pass  # TODO: run one learning episode

    return Q

In [ ]:
print("Exercise 5 checks")
Q_zero = q_learning(0, seed=1)
print("✅ correct state count" if len(Q_zero) == 16 else "🟠 expected 16 states")
print("✅ four actions per state" if all(set(Q_zero[s]) == set(ACTIONS) for s in STATES)
      else "🟠 each state needs all four actions")
check_close("terminal Q(0,up)", Q_zero[0]["up"], 0.0)
check_close("terminal Q(15,left)", Q_zero[15]["left"], 0.0)

Q_short = q_learning(80, alpha=0.1, gamma=1.0, seed=11)
learned_something = any(Q_short[s][a] < 0 for s in STATES if s not in TERMINALS for a in ACTIONS)
print("✅ repeated episodes changed nonterminal Q values" if learned_something
      else "🟠 no learning yet — check the episode call and earlier exercises")

## Final experiment

Once every checker is green, run TD(0) for more episodes and inspect all four Q values at each state. Actions that move toward a nearby terminal should generally be less negative.

In [ ]:
Q_learned = q_learning(10_000, alpha=0.05, gamma=1.0, seed=21)
show_q_values(Q_learned)

<div style="border-left:6px solid #a855f7;background:#faf5ff;padding:12px 16px;border-radius:8px">
<b>Interpret before moving on</b>
<ol style="margin-bottom:0">
<li>Why can TD(0) update before the episode finishes?</li>
<li>Why does the target use the maximum next-state Q value?</li>
<li>Why is the next-state estimate zero on a terminal transition?</li>
<li>What happens when the TD error is positive? What if it is negative?</li>
<li>Try <code>alpha=0.5</code> and <code>alpha=0.01</code>. Which looks noisier after 10,000 episodes?</li>
</ol>
</div>

**Completion test:** you can explain the max target, TD error, and Q update; all five checker sections pass; and you can trace one transition by hand.